# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
md = dataset.metadata
print(f"Title: {md.name}\nDescription: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# List all record sets by their @id
record_sets = list(dataset.metadata.record_sets)
print('Record sets (by @id):')
for rs in record_sets:
    print(f"- {rs['@id']}")

# For each record set, show its fields (by @id)
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print('  Fields:')
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            else:
                print(f"    - {f}")
    else:
        print('  No fields listed.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Available record set @ids:", record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print("  No records found.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Number of records loaded: {len(df)}")
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(2))

# For demonstration, pick the first record set (if any records were loaded)
if dataframes:
    main_record_set = list(dataframes.keys())[0]
    print(f"\nUsing record set @id: {main_record_set}")
    print(f"Columns: {dataframes[main_record_set].columns.tolist()}")
    print(dataframes[main_record_set].head())
else:
    print("No dataframes loaded. Check if the record sets contain records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section applies operations such as removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
if dataframes:
    df = dataframes[main_record_set]
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print("Available numeric fields:", numeric_candidates)
    
    # Pick the first numeric field for demonstration
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        threshold = df[numeric_field].quantile(0.75)  # use 75th percentile as threshold for illustration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # If a string field for grouping exists, use the first non-numeric as group_field
        group_candidates = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable group_field found for grouping.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No loaded DataFrames. Cannot perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if dataframes and 'numeric_field' in locals():
    # Histogram of the numeric field
    df = dataframes[main_record_set]
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
    
    # If a group_field was found, plot mean per group
    if 'group_field' in locals():
        grouped = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        grouped.plot(kind='bar', figsize=(10,4))
        plt.ylabel(f'Mean {numeric_field}')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.show()
else:
    print("No numeric field to visualize. Run the EDA section first.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains detailed protein abundance and modification data from mass spectrometry analysis of human mast cell extracellular vesicles.
- Using `mlcroissant`, you can load record sets by their `@id` and process fields by `@id` for flexible, schema-driven data analysis.
- Initial EDA demonstrates filtering, normalization, and aggregation/grouping by string fields as suitable.
- Visualization shows distributions and group differences, supporting downstream biomarker/biological insights.
